<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/MLP_code_5_fold_Cross_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------------------------
# 1. Input Data & Reproducibility
# -------------------------------------------------

tf.keras.utils.set_random_seed(42)

data = pd.read_csv('/content/GTPP-DATASET.csv')

X = data[['AT', 'AP', 'DF', 'CH', 'GP', 'CPR', 'CPD', 'TTXM']].values
y = data['EP'].values

# -------------------------------------------------
# 2. Holdout Split (80% Train / 20% Untouched Test)
# -------------------------------------------------

X_train_full, X_test_holdout, y_train_full, y_test_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# =================================================
# SUB-STEP 4A: 5-FOLD CV INSIDE 80% TRAINING SET
# =================================================

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
optimal_epochs = []

print("Starting Sub-Step 4A: 5-Fold Cross-Validation on 80% Training Set...\n")

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train_full), start=1):

    # 4 chunks for training, 1 chunk for validation (NO validation_split needed)
    X_tr, X_val = X_train_full[train_idx], X_train_full[val_idx]
    y_tr, y_val = y_train_full[train_idx], y_train_full[val_idx]

    # Fit MinMaxScaler strictly on fold's training split
    scaler = MinMaxScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    tf.keras.utils.set_random_seed(42)

    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(8,)),
        tf.keras.layers.Dense(224, activation='tanh'),
        tf.keras.layers.Dense(48, activation='tanh'),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001603),
        loss='mse',
        metrics=['mae']
    )

    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=12,
        min_lr=1e-6,
        verbose=0
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=25,
        restore_best_weights=True
    )

    history = model.fit(
        X_tr_scaled,
        y_tr,
        epochs=200,
        batch_size=16,
        validation_data=(X_val_scaled, y_val),
        callbacks=[lr_scheduler, early_stop],
        verbose=0
    )

    best_ep = np.argmin(history.history['val_loss']) + 1
    optimal_epochs.append(best_ep)

    y_val_pred = model.predict(X_val_scaled, verbose=0).flatten()

    mae = mean_absolute_error(y_val, y_val_pred)
    mse = mean_squared_error(y_val, y_val_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_val_pred)

    fold_results.append({
        "Fold": f"Fold {fold}",
        "Best Epoch": best_ep,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    print(f"Fold {fold} (Epoch {best_ep:3d}) -> MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.6f}")

cv_df = pd.DataFrame(fold_results)

print("\n==========================================================================")
print("          SUB-STEP 4A: 5-FOLD CROSS-VALIDATION SUMMARY (80% TRAIN)        ")
print("==========================================================================")
print(cv_df.to_string(index=False))
print("--------------------------------------------------------------------------")
print(f"Mean MAE      : {cv_df['MAE'].mean():.6f} +/- {cv_df['MAE'].std():.6f}")
print(f"Mean RMSE     : {cv_df['RMSE'].mean():.6f} +/- {cv_df['RMSE'].std():.6f}")
print(f"Mean R2 Score : {cv_df['R2 Score'].mean():.6f} +/- {cv_df['R2 Score'].std():.6f}")
print("==========================================================================\n")

# =================================================
# SUB-STEP 4B: FINAL FIT ON FULL 80% & TEST ON 20%
# =================================================

final_scaler = MinMaxScaler()
X_train_full_scaled = final_scaler.fit_transform(X_train_full)
X_test_holdout_scaled = final_scaler.transform(X_test_holdout)

tf.keras.utils.set_random_seed(42)

final_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(8,)),
    tf.keras.layers.Dense(160, activation='tanh'),
    tf.keras.layers.Dense(48, activation='tanh'),
    tf.keras.layers.Dense(1)
])

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001603),
    loss='mse',
    metrics=['mae']
)

lr_scheduler_final = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=12,
    min_lr=1e-6,
    verbose=0
)

final_model.fit(
    X_train_full_scaled,
    y_train_full,
    epochs=200,
    batch_size=16,
    callbacks=[lr_scheduler_final],
    verbose=0
)

y_test_pred = final_model.predict(X_test_holdout_scaled, verbose=0).flatten()

test_mae = mean_absolute_error(y_test_holdout, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_holdout, y_test_pred))
test_r2 = r2_score(y_test_holdout, y_test_pred)

print("==========================================================================")
print("         SUB-STEP 4B: FINAL UNTOUCHED 20% HOLDOUT TEST EVALUATION         ")
print("==========================================================================")
print(f"Holdout Test MAE      : {test_mae:.6f}  (Paper SVR: 0.847000)")
print(f"Holdout Test RMSE     : {test_rmse:.6f}  (Paper SVR: 1.187000)")
print(f"Holdout Test R2 Score : {test_r2:.6f}  (Paper SVR: 0.966380)")
print("==========================================================================")

Starting Sub-Step 4A: 5-Fold Cross-Validation on 80% Training Set...

Fold 1 (Epoch 200) -> MAE: 0.9112 | RMSE: 1.1732 | R2: 0.965381
Fold 2 (Epoch  49) -> MAE: 0.9519 | RMSE: 1.2380 | R2: 0.962107
Fold 3 (Epoch 192) -> MAE: 0.9485 | RMSE: 1.2540 | R2: 0.954914
Fold 4 (Epoch 200) -> MAE: 0.9492 | RMSE: 1.2504 | R2: 0.961657
Fold 5 (Epoch  65) -> MAE: 0.9615 | RMSE: 1.2580 | R2: 0.955803

          SUB-STEP 4A: 5-FOLD CROSS-VALIDATION SUMMARY (80% TRAIN)        
  Fold  Best Epoch      MAE     RMSE  R2 Score
Fold 1         200 0.911156 1.173156  0.965381
Fold 2          49 0.951879 1.238047  0.962107
Fold 3         192 0.948466 1.253977  0.954914
Fold 4         200 0.949217 1.250363  0.961657
Fold 5          65 0.961550 1.258023  0.955803
--------------------------------------------------------------------------
Mean MAE      : 0.944453 +/- 0.019332
Mean RMSE     : 1.234713 +/- 0.035213
Mean R2 Score : 0.959972 +/- 0.004461

         SUB-STEP 4B: FINAL UNTOUCHED 20% HOLDOUT TEST EVALUAT